# 셀 1: 라이브러리 임포트

In [1]:
import os
from datetime import datetime
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
from tqdm.auto import tqdm
from collections import Counter

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel

# 성능 관련 환경 변수 / 백엔드 설정
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


/home/team036/.local/lib/python3.10/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


# 셀 2: 모델 선택 (여기만 수정하세요!)


In [2]:
# 모델 자동 선택 방식:
#   /home/team036/content/Qwen3-VL-4B-finetuned_YYYYMMDD_HHMMSS
# 형식으로 저장된 가장 최신 디렉토리를 자동으로 고른다.
USE_LATEST = True
# SPECIFIC_VERSION = "20241026_143022"  # 특정 버전 쓰고 싶으면 USE_LATEST=False 하고 여기에 맞춰 입력

BASE_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
BASE_SAVE_DIR = "/home/team036/content"

IMAGE_SIZE = 224        # 학습과 동일해야 함 (중요)
BATCH_SIZE = 32         # 가능한 크게: GPU VRAM 여유 보고 조정
NUM_WORKERS = 0         # 이미지 로드/리사이즈용 CPU worker

TEST_CSV = "/home/team036/aichallenge/test.csv"

OUTPUT_DIR = "/home/team036/aichallenge/submissions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 셀 3: 모델 버전 선택 (timestamp 기반)

In [3]:
def get_available_models(base_dir):
    """
    base_dir 아래에서
    Qwen3-VL-4B-finetuned_YYYYMMDD_HHMMSS
    형태의 디렉토리를 찾아서 정렬된 리스트로 반환.
    """
    models = []
    if not os.path.exists(base_dir):
        return models
    
    for item in os.listdir(base_dir):
        full_path = os.path.join(base_dir, item)
        if item.startswith("Qwen3-VL-4B-finetuned_") and os.path.isdir(full_path):
            version = item.replace("Qwen3-VL-4B-finetuned_", "")
            models.append({
                "version": version,      # "YYYYMMDD_HHMMSS"
                "path": full_path,       # /home/team036/content/Qwen3-VL-4B-finetuned_...
                "full_name": item
            })
    # 버전(timestamp 문자열) 기준 정렬
    models.sort(key=lambda x: x["version"])
    return models


def print_available_models():
    models = get_available_models(BASE_SAVE_DIR)
    if not models:
        print("⚠️  저장된 fine-tuned 모델이 없습니다.")
        return None
    
    print(f"\n{'='*60}")
    print("📦 사용 가능한 모델 버전들:")
    print(f"{'='*60}")
    for i, m in enumerate(models, 1):
        info_file = os.path.join(m["path"], "training_info.txt")
        print(f"\n[{i}] Version: {m['version']}")
        print(f"    Path: {m['path']}")
        if os.path.exists(info_file):
            with open(info_file, "r") as f:
                for line in f:
                    if ("Validation Loss" in line
                        or "Validation Acc" in line
                        or "Training Date" in line):
                        print(f"    {line.strip()}")
    print(f"\n{'='*60}\n")
    return models


available_models = print_available_models()
if available_models is None or len(available_models) == 0:
    raise RuntimeError("❌ Fine-tuned 모델이 없습니다. 먼저 학습을 진행해주세요.")

if USE_LATEST:
    selected_model = available_models[-1]
    print(f"✓ 최신 모델 선택: {selected_model['version']}")
else:
    selected_model = None
    for m in available_models:
        if m["version"] == SPECIFIC_VERSION:
            selected_model = m
            break
    if selected_model is None:
        raise RuntimeError(f"❌ 버전 '{SPECIFIC_VERSION}'을 찾을 수 없습니다.")
    print(f"✓ 선택된 모델: {selected_model['version']}")

FINETUNED_MODEL_DIR = selected_model["path"]   # LoRA 어댑터 + processor 저장된 폴더
MODEL_VERSION       = selected_model["version"]


📦 사용 가능한 모델 버전들:

[1] Version: 20251026_044259
    Path: /home/team036/content/Qwen3-VL-4B-finetuned_20251026_044259
    Training Date: 2025-10-26 06:58:53
    Validation Loss: 2.5596


✓ 최신 모델 선택: 20251026_044259


# 셀 3: 프롬프트 생성 함수 (학습과 완전히 동일하게)

In [4]:

SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

import re
def clean_text(x: str):
    x = str(x)
    x = x.replace("\t", " ").replace("\n", " ").strip()
    x = re.sub(r"\s+", " ", x)
    return x

def build_mc_prompt(question, a, b, c, d):
    question = clean_text(question)
    a = clean_text(a); b = clean_text(b); c = clean_text(c); d = clean_text(d)
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Output only the correct answer as a single lowercase letter."
    )

def extract_choice(text: str) -> str:
    """
    모델 출력에서 최종적으로 a/b/c/d 한 글자만 뽑는다.
    """
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if lines:
        last = lines[-1]
        if last in ["a", "b", "c", "d"]:
            return last
        for tok in last.split():
            if tok in ["a", "b", "c", "d"]:
                return tok
    for ch in ["a", "b", "c", "d"]:
        if ch in text:
            return ch
    return "a"  # fallback 안전빵

# 셀 4: Fine-tuned 모델 로드 (bf16 일관)

In [5]:
print(f"\nLoading fine-tuned model: {MODEL_VERSION} ...")

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,  # 학습과 동일하게 bf16 경로 사용
# )

# 1) 베이스 모델 로드 (4bit)
# base_model = Qwen3VLForConditionalGeneration.from_pretrained(
#     BASE_MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
# )

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    # quantization_config=bnb_config,  # 🛑 [제거]
    torch_dtype=torch.bfloat16,      # 🔥 [변경/추가] bfloat16으로 바로 로드
    device_map="auto",
    trust_remote_code=True,
)

# 2) Fine-tuned LoRA 어댑터 로드
lora_model = PeftModel.from_pretrained(
    base_model,
    FINETUNED_MODEL_DIR,
    device_map="auto" # 🔥 [추가] 안정성을 위해 추가
)
print("✓ Fine-tuned LoRA adapter loaded")

# 3) LoRA 가중치를 base_model에 merge해서 추론 전용 단일 모델로 만들기
model = lora_model.merge_and_unload()
model.eval()
for p in model.parameters():
    p.requires_grad = False

# 🔥 [추가] torch.compile로 모델 컴파일
# 컴파일에 1~2분 정도 소요될 수 있지만, 이후 추론 속도가 매우 빨라집니다.
print("Compiling model... (This may take a minute)")
try:
    model = torch.compile(model, mode="reduce-overhead")
    print("✓ Model compiled successfully")
except Exception as e:
    print(f"⚠️ Model compilation failed: {e}. Running without compile.")

# 4) 캐시 사용 강제 (generate에서 past_key_values 쓰도록)
model.config.use_cache = True
if hasattr(model, "base_model") and hasattr(model.base_model, "config"):
    model.base_model.config.use_cache = True

# 5) Processor 로드
processor = AutoProcessor.from_pretrained(
    FINETUNED_MODEL_DIR,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "left"

print(f"✓ Processor loaded (padding_side={processor.tokenizer.padding_side})")
print("✓ Model ready for inference")


Loading fine-tuned model: 20251026_044259 ...


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.42s/it]


✓ Fine-tuned LoRA adapter loaded
Compiling model... (This may take a minute)
✓ Model compiled successfully
✓ Processor loaded (padding_side=left)
✓ Model ready for inference


# 셀 5: 테스트 데이터 로드

In [6]:
test_df = pd.read_csv(TEST_CSV)
print(f"\n✓ Test samples: {len(test_df)}")


✓ Test samples: 3887


# 셀 6: Dataset 및 Collator 정의 (inference에서도 resize=224)

In [7]:

class TestDataset(Dataset):
    """
    - test_df[row]:
        - path: 이미지 경로
        - question, a, b, c, d
    - 이미지: 224x224로 리사이즈 (학습과 동일)
    - user_text: multiple choice prompt
    """
    def __init__(self, df, image_size=224):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        user_text = build_mc_prompt(
            row["question"], row["a"], row["b"], row["c"], row["d"]
        )
        return img, user_text


class TestCollator:
    """
    - 배치(list[(PIL.Image, prompt_text)])
    - 메시지를 Qwen3-VL chat 형식으로 만들고
      processor.apply_chat_template -> text
    - processor(...)로 텐서화 (input_ids, pixel_values 등)
    """
    def __init__(self, processor, system_instruct):
        self.processor = processor
        self.system_instruct = system_instruct
    
    def __call__(self, batch):
        images, user_texts = zip(*batch)
        images = list(images)

        texts = []
        for img, user_text in zip(images, user_texts):
            messages = [
                {"role": "system", "content": [
                    {"type": "text", "text": self.system_instruct}
                ]},
                {"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text",  "text": user_text}
                ]}
            ]
            # generation 프롬프트를 붙여서 바로 답변을 생성하도록 유도
            templated = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            texts.append(templated)
        
        batch_inputs = self.processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
        )
        return batch_inputs

print("✓ Dataset and Collator defined")

✓ Dataset and Collator defined


# 셀 7: DataLoader 생성

In [8]:
test_dataset = TestDataset(test_df, image_size=IMAGE_SIZE)
test_collator = TestCollator(processor, SYSTEM_INSTRUCT)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=test_collator,
    pin_memory=True,
)

print("✓ DataLoader created")


✓ DataLoader created


# 셀 8: 배치 추론 실행

In [ ]:
predictions = []

print(f"\n{'='*60}")
print(f"Starting inference with model version: {MODEL_VERSION}")
print(f"{'='*60}\n")

with tqdm(total=len(test_df), desc="Inference", unit="sample") as pbar:
    for batch_idx, inputs in enumerate(test_loader):
        # inputs: dict of tensors from processor
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

        # 현재 시퀀스 길이 저장: 새 토큰만 추출하기 위함
        input_length = inputs["input_ids"].shape[1]

        # bf16 autocast + no_grad로 빠르게
        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.bfloat16): # with torch.no_grad()
            output_ids = model.generate(
                **inputs,
                max_new_tokens=2,               # 정답 한 글자만 필요하므로 매우 짧게
                do_sample=False,                # greedy
                eos_token_id=processor.tokenizer.eos_token_id,
                pad_token_id=processor.tokenizer.pad_token_id,
            )

        # 새로 생성된 부분만 떼기
        gen_ids = output_ids[:, input_length:]

        # 디코드
        output_texts = processor.batch_decode(
            gen_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        # 첫 배치 디버깅
        if batch_idx == 0:
            tqdm.write("\n=== 첫 번째 배치 디버깅 ===")
            for i, out_txt in enumerate(output_texts[:3]):
                tqdm.write(f"[샘플 {i}] 생성: {repr(out_txt)} → 추출: {extract_choice(out_txt)}")
            tqdm.write("="*60 + "\n")

        # 후처리해서 a/b/c/d 뽑기
        batch_preds = [extract_choice(t) for t in output_texts]
        predictions.extend(batch_preds)

        # 진행률 업데이트
        pbar.update(len(output_texts))



Starting inference with model version: 20251026_044259



Inference:   0%|          | 0/3887 [00:00<?, ?sample/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


# 셀 9: 결과 저장


In [ ]:
print(f"\n{'='*60}")
print("Inference Results")
print(f"{'='*60}")
print(f"예측 개수: {len(predictions)}")
print(f"테스트 데이터 개수: {len(test_df)}")

# 분포 출력 (혹시 한 글자만 몰빵하는지 확인)
pred_distribution = Counter(predictions)
print("\n예측 분포:")
for choice, count in sorted(pred_distribution.items()):
    pct = (count / len(predictions)) * 100
    print(f"  {choice}: {count} ({pct:.1f}%)")

# 제출 CSV 저장
submission_filename = f"submission_{MODEL_VERSION}.csv"
submission_path = os.path.join(OUTPUT_DIR, submission_filename)

submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": predictions,
})
submission.to_csv(submission_path, index=False)

print(f"\n✓ Saved: {submission_path}")
print(f"  Model Version: {MODEL_VERSION}")
print(f"  Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}\n")

torch.cuda.empty_cache()
print("✓ GPU memory cleared")